# Kemet — Google Maps Beaches Scraper

In [1]:
!pip install selenium pandas -q


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [3]:
places_to_scrape = [
    # The Mediterranean & North Coast
    "Ageeba Beach, Marsa Matrouh", "Cleopatra Beach, Marsa Matrouh",
    "Rommel Beach, Marsa Matrouh", "Gharam Beach, Marsa Matrouh",
    "Almaza Bay Beach, North Coast", "Marassi Beach, North Coast",
    "Hacienda White Beach, North Coast", "Mamoura Beach, Alexandria",
    "Montazah Beach, Alexandria", "Citadel of Qaitbay Waterfront, Alexandria",

    # South Sinai
    "Ras Mohammed National Park, Sharm El Sheikh", "Naama Bay Beach, Sharm El Sheikh",
    "Shark's Bay Beach, Sharm El Sheikh", "Ras Um Sid Beach, Sharm El Sheikh",
    "Terrazzina Beach, Sharm El Sheikh", "Nabq Nature Reserve, Sharm El Sheikh",
    "Blue Hole, Dahab", "Three Pools, Dahab", "Laguna Beach, Dahab",
    "Eel Garden Reef, Dahab", "Lighthouse Dive Site, Dahab",
    "Ras Abu Galum Reserve, Nuweiba", "Castle Zaman Private Beach, Nuweiba",
    "Fjord Bay, Taba", "Pharoah's Island, Taba",

    # The Red Sea
    "Mahmya Island, Hurghada", "Giftun Island National Park, Hurghada",
    "Orange Bay, Hurghada", "Makadi Bay Water World, Hurghada",
    "Sahl Hasheesh Beach, Hurghada", "Zeytouna Beach, El Gouna",
    "Mangroovy Beach, El Gouna", "Sliders Cable Park, El Gouna",
    "Soma Bay, Safaga", "Sharm El Naga Resort Beach, Safaga",
    "Abu Nuhas Shipwreck Dive Site, Red Sea",

    # The Deep Red Sea South
    "Sharm El Luli, Marsa Alam", "Abu Dabbab Beach, Marsa Alam",
    "Wadi El Gemal National Park Coast, Marsa Alam", "Satayah Dolphin Reef, Marsa Alam",
    "Marsa Mubarak, Marsa Alam", "Nayzak Beach, Marsa Alam",
    "Samadai Reef, Marsa Alam", "Hamata Islands, Marsa Alam",
    "El Quseir Fort Beach area, El Quseir",

    # Unique Inland & River Water Activities
    "Magic Lake, Wadi El Rayan, Fayoum", "Wadi El Rayan Waterfalls, Fayoum",
    "Lake Qarun Waterfront, Fayoum", "Elephantine Island Felucca Sailing, Aswan",
    "Burullus Lake, Kafr El Sheikh",
]
print(len(places_to_scrape), "places")

50 places


In [8]:
driver = webdriver.Chrome()
results = []

for place in places_to_scrape:
    print("Scraping:", place)
    driver.get("https://www.google.com/maps/search/" + place.replace(" ", "+") + "?hl=en")

    try:
        WebDriverWait(driver, 15).until(
            lambda d: d.find_elements(By.CSS_SELECTOR, "h1.DUwDvf")
            or d.find_elements(By.CSS_SELECTOR, "div.Nv2PK")
        )
    except:
        pass

    if driver.find_elements(By.CSS_SELECTOR, "div.Nv2PK"):
        try:
            first_result = driver.find_element(By.CSS_SELECTOR, "div.Nv2PK a.hfpxzc")
            first_result.click()
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1.DUwDvf"))
            )
        except:
            pass

    time.sleep(1.5)

    try:
        name = driver.find_element(By.CSS_SELECTOR, "h1.DUwDvf").text
    except:
        name = "N/A"

    try:
        category = driver.find_element(By.CSS_SELECTOR, "button.DkEaL").text
    except:
        category = "N/A"

    try:
        rating = driver.find_element(By.CSS_SELECTOR, "div.F7nice span[aria-hidden='true']").text
    except:
        rating = "N/A"

    try:
        description = driver.find_element(By.CSS_SELECTOR, "div.PYvSYb").text
    except:
        description = "N/A"

    dogs_allowed = "Yes" if "dog" in description.lower() else "N/A"

    try:
        address = driver.find_element(By.CSS_SELECTOR, "button[data-item-id='address']").get_attribute("aria-label")
        address = address.replace("Address:", "").strip() if address else "N/A"
    except:
        address = "N/A"

    # Improved photo extraction
    photo_url = "N/A"
    selectors = [
        "button.aoRNLd img",
        "div.RZ66Rb img",
        "button[jsaction*='heroHeaderImage'] img"
    ]

    for sel in selectors:
        try:
            img = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, sel))
            )

            src = img.get_attribute("src")
            srcset = img.get_attribute("srcset")

            if src:
                photo_url = src
                break
            elif srcset:
                photo_url = srcset.split(",")[-1].split(" ")[0]
                break

        except:
            continue

    # fallback if selectors fail
    if photo_url == "N/A":
        try:
            imgs = driver.find_elements(By.TAG_NAME, "img")
            for img in imgs:
                src = img.get_attribute("src")
                if src and "googleusercontent" in src:
                    photo_url = src
                    break
        except:
            pass

    maps_url = driver.current_url

    results.append({
        "Name": name,
        "Category": category,
        "Rating": rating,
        "Description": description,
        "Dogs_Allowed": dogs_allowed,
        "Address": address,
        "Photo_URL": photo_url,
        "Maps_URL": maps_url,
    })

    time.sleep(2)

driver.quit()

Scraping: Ageeba Beach, Marsa Matrouh
Scraping: Cleopatra Beach, Marsa Matrouh
Scraping: Rommel Beach, Marsa Matrouh
Scraping: Gharam Beach, Marsa Matrouh
Scraping: Almaza Bay Beach, North Coast
Scraping: Marassi Beach, North Coast
Scraping: Hacienda White Beach, North Coast
Scraping: Mamoura Beach, Alexandria
Scraping: Montazah Beach, Alexandria
Scraping: Citadel of Qaitbay Waterfront, Alexandria
Scraping: Ras Mohammed National Park, Sharm El Sheikh
Scraping: Naama Bay Beach, Sharm El Sheikh
Scraping: Shark's Bay Beach, Sharm El Sheikh
Scraping: Ras Um Sid Beach, Sharm El Sheikh
Scraping: Terrazzina Beach, Sharm El Sheikh
Scraping: Nabq Nature Reserve, Sharm El Sheikh
Scraping: Blue Hole, Dahab
Scraping: Three Pools, Dahab
Scraping: Laguna Beach, Dahab
Scraping: Eel Garden Reef, Dahab
Scraping: Lighthouse Dive Site, Dahab
Scraping: Ras Abu Galum Reserve, Nuweiba
Scraping: Castle Zaman Private Beach, Nuweiba
Scraping: Fjord Bay, Taba
Scraping: Pharoah's Island, Taba
Scraping: Mahmya Is

In [9]:
df = pd.DataFrame(results)
df.to_csv("kemet_beaches_data.csv", index=False, encoding="utf-8-sig")
df.head(10)

,Name,Category,Rating,Description,Dogs_Allowed,Address,Photo_URL,Maps_URL
0,Ageeba Beach,Public beach,4.6,Small cove accessible via a path down from the...,N/A,"C274+PHF شاطىء عجيبة, Mersa Matruh, Marsa Matr...",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/place/Ageeba+Beach...
1,Cleopatra Beach,Beach,4.3,Famous rocky beach with a statue of Cleopatra ...,N/A,"Mersa Matruh, Marsa Matrouh Governorate",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/search/Cleopatra+B...
2,⛱Rommel Beach,Beach,4.1,N/A,N/A,"Mersa Matruh, Marsa Matrouh Governorate 5057083",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/search/Rommel+Beac...
3,lover Beach,Beach,4.2,N/A,N/A,"Mersa Matruh, Marsa Matrouh Governorate 5053296",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/place/lover+Beach/...
4,Almaza Bay,N/A,4.7,N/A,N/A,"Unnamed Road,, Mersa Matruh, Marsa Matrouh Gov...",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/place/Almaza+Bay/@...
5,North Beach - Marassi,Tourist attraction,4.5,N/A,N/A,"El Alamein, Marsa Matrouh Governorate 5065501",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/search/Marassi+Bea...
6,Hacienda White - Palm Hills Developments,Beach,4.5,N/A,N/A,"KM 137 Alexandria - Marsa Matrouh Rd, Al Dabaa...",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/place/Hacienda+Whi...
7,El Mamurah Beach,Public beach,4.2,Popular sunbathing beach with seafront eaterie...,N/A,"72RJ+87F, Street 17, Al Mandarah Bahri, Montaz...",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/place/El+Mamurah+B...
8,El Montazah Beach,Beach,4.2,N/A,N/A,"Al Mandarah Bahri, Montaza 2, Alexandria Gover...",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/place/El+Montazah+...
9,Qaitbay Citadel,Castle,4.5,N/A,N/A,"السيالة شرق، قسم الجمرك،, Alexandria Governorate",https://lh3.googleusercontent.com/gps-cs-s/APN...,https://www.google.com/maps/place/Qaitbay+Cita...
